In [25]:
# STEP 1 — Overview: Script Purpose and Workflow
# ------------------------------------------------
# This script updates ESP-r "pv" configuration files for each building
# using the MODELED_PV values from your EPC dataset.
#
# What it does:
# 1. Loads the EPC dataset CSV file.
# 2. Iterates through each building using BUILDING_REFERENCE_NUMBER.
# 3. Locates the corresponding building folder in "baseline_models".
# 4. Finds the model subfolder (e.g., Detached_2F, Flat, etc.).
# 5. Navigates to the "cfg/pv" file.
# 6. Updates the 9th value in the PV data line:
#    → "Number of panels in surface"
#    → Sets it equal to MODELED_PV for that building.
# 7. Skips any buildings where the "pv" file is missing.
# 8. Logs all updates and skipped cases.
#
# Output:
# - Updated pv files in-place
# - Console log of processed / skipped buildings

import os
import pandas as pd

In [ ]:
# STEP 2 — Load Dataset
# ------------------------------------------------
# Reads the EPC dataset and prepares key columns

csv_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"

df = pd.read_csv(csv_path)

# Ensure correct types
df["BUILDING_REFERENCE_NUMBER"] = df["BUILDING_REFERENCE_NUMBER"].astype(str)
df["MODELED_PV"] = pd.to_numeric(df["MODELED_PV"], errors="coerce")

# Drop rows without valid PV values
df = df.dropna(subset=["MODELED_PV"])

print(f"Loaded dataset with {len(df)} valid entries.")

Loaded dataset with 117 valid entries.


In [ ]:
# STEP 3 — Define Base Directory and Model Folder Options
# ------------------------------------------------

base_dir = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

model_folder_names = [
    "SemiDetached_2F", "Detached_2F"
]

In [28]:
# STEP 4 (UPDATED v2) — Precise PV Data Block Parser
# ------------------------------------------------
# Fixes:
# - Reliably finds "# Data:" section (even with messy headers)
# - Skips all comment lines safely
# - Explicitly targets the FIRST numeric data line after "# Data:"
# - Ensures correct update of the 9th value (index 8)
# - Avoids false matches from earlier numeric lines in file

def update_pv_file(pv_path, new_value, debug=False):
    try:
        with open(pv_path, "r") as f:
            lines = f.readlines()

        # Step 1: Find "# Data:" line
        data_idx = None
        for i, line in enumerate(lines):
            if "# Data:" in line:
                data_idx = i
                break

        if data_idx is None:
            return False, "No '# Data:' section found"

        # Step 2: Find first numeric line AFTER "# Data:"
        first_data_line_idx = None
        for i in range(data_idx + 1, len(lines)):
            line = lines[i].strip()

            # Skip comments and empty lines
            if not line or line.startswith("#"):
                continue

            parts = line.split()

            # Check if line is numeric
            try:
                floats = [float(x) for x in parts]
                if len(floats) >= 9:
                    first_data_line_idx = i
                    break
            except:
                continue

        if first_data_line_idx is None:
            return False, "Could not find first numeric data line"

        # Step 3: Process that line
        original_line = lines[first_data_line_idx]
        parts = original_line.split()

        if len(parts) < 9:
            return False, "Not enough values in PV data line"

        if debug:
            print(f"\nFile: {pv_path}")
            print(f"Before: {parts}")

        # Step 4: Update 9th value
        parts[8] = f"{float(new_value):.4f}"

        if debug:
            print(f"After:  {parts}")

        # Step 5: Rebuild line (preserve general spacing)
        new_line = "   " + "   ".join(parts) + "\n"
        lines[first_data_line_idx] = new_line

        # Step 6: Write back
        with open(pv_path, "w") as f:
            f.writelines(lines)

        return True, None

    except Exception as e:
        return False, str(e)

In [29]:
# STEP 5 (UPDATED) — Main Loop with Debug Option
# ------------------------------------------------
# Adds optional debug printing for first few buildings

updated_count = 0
skipped_missing_pv = 0
skipped_no_model = 0
errors = []

DEBUG_FIRST_N = 3  # set to 0 to disable debug output

for i, (_, row) in enumerate(df.iterrows()):
    building_id = row["BUILDING_REFERENCE_NUMBER"]
    pv_value = row["MODELED_PV"]

    building_path = os.path.join(base_dir, building_id)

    if not os.path.isdir(building_path):
        skipped_no_model += 1
        continue

    # Find model folder
    model_folder = None
    for name in model_folder_names:
        candidate = os.path.join(building_path, name)
        if os.path.isdir(candidate):
            model_folder = candidate
            break

    if model_folder is None:
        skipped_no_model += 1
        continue

    pv_path = os.path.join(model_folder, "cfg", "pv")

    if not os.path.isfile(pv_path):
        skipped_missing_pv += 1
        continue

    debug_mode = i < DEBUG_FIRST_N

    success, err = update_pv_file(pv_path, pv_value, debug=debug_mode)

    if success:
        updated_count += 1
    else:
        errors.append((building_id, err))

In [30]:
# STEP 6 — Summary Output
# ------------------------------------------------

print("=== PROCESSING COMPLETE ===")
print(f"Updated PV files: {updated_count}")
print(f"Skipped (missing pv file): {skipped_missing_pv}")
print(f"Skipped (no model folder): {skipped_no_model}")
print(f"Errors: {len(errors)}")

# Optional: print errors
for bldg, err in errors[:10]:
    print(f"Error in {bldg}: {err}")

=== PROCESSING COMPLETE ===
Updated PV files: 1
Skipped (missing pv file): 115
Skipped (no model folder): 1
Errors: 0
